# Tabla orders

## Fechas como texto

In [164]:
import pandas as pd
from pathlib import Path
import numpy as np

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")

In [54]:
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
orders.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Vemos que las columnas que deberían mostrar fechas son texto (object). Vamos a cambiarlas a tipo date.

In [57]:
columnas_fecha = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                  "order_delivered_customer_date", "order_estimated_delivery_date"]

for col in columnas_fecha:
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## Pedidos con fechas incoherentes:

En la exploración vimos dos tipos de fechas sospechosas:

- 8 pedidos delivered sin fecha de entrega al cliente.
- 23 pedidos con la entrega al cliente anterior a la del transportista, algo imposible: no puede llegar a tu casa antes de salir del almacén.

Vamos a marcarlos, no borrarlos.

In [72]:
# Condición 1: pedido "delivered" pero sin fecha de entrega al cliente
sin_fecha_entrega = (orders.order_status == "delivered") & orders.order_delivered_customer_date.isna()

# Condición 2: la entrega al cliente es anterior a la entrega al transportista
entrega_antes_de_salir = orders.order_delivered_customer_date < orders.order_delivered_carrier_date

# Nueva columna: True si se cumple alguna de las dos ("|" significa "o")
orders["fechas_incoherentes"] = sin_fecha_entrega | entrega_antes_de_salir

print(sin_fecha_entrega.sum(), entrega_antes_de_salir.sum(), orders.fechas_incoherentes.sum())

8 23 31


Verificamos que no perdimos nada:

In [75]:
print(len(orders))                       # debe seguir siendo 99441
orders.groupby("order_status").fechas_incoherentes.sum()

99441


order_status
approved        0
canceled        0
created         0
delivered      31
invoiced        0
processing      0
shipped         0
unavailable     0
Name: fechas_incoherentes, dtype: int64

Guardamos la tabla limpia:

In [78]:
orders.to_csv(CLEAN / "orders.csv", index=False)

In [101]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,fechas_incoherentes
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False


# Tabla items

In [84]:
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
items.dtypes

order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

In [86]:
columnas_fecha = ["shipping_limit_date"]

for col in columnas_fecha:
    items[col] = pd.to_datetime(items[col])

items.dtypes

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

In [93]:
items.to_csv(CLEAN / "items.csv", index=False)

# Tabla reviews

## Fechas como texto

In [89]:
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
reviews.dtypes

review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

In [91]:
columnas_fecha = ["review_creation_date", "review_answer_timestamp"]

for col in columnas_fecha:
    reviews[col] = pd.to_datetime(reviews[col])

reviews.dtypes

review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

## Una reseña por pedido

In [107]:
reviews = reviews.sort_values(["order_id", "review_answer_timestamp"])
reviews = reviews.drop_duplicates(subset="order_id", keep="last")

- sort_values([...]) ordena la tabla primero por order_id y, dentro de cada pedido, por fecha de respuesta de más antigua a más reciente.

- drop_duplicates(subset="order_id", keep="last") elimina las filas cuyo order_id ya apareció antes, conservando la última de cada pedido. Al estar ordenadas por fecha, la última es la más reciente.

In [110]:
print(len(reviews))                # debe salir 98673
print(reviews.order_id.is_unique)  # debe salir True

98673
True


In [112]:
reviews.to_csv(CLEAN / "reviews.csv" , index=False)

# Tabla payments

In [119]:
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
payments.dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [125]:
no_definidos = payments[payments.payment_type == "not_defined"]
print(no_definidos)

# ¿en qué estado están esos pedidos? (orders es la tabla limpia que ya tienes cargada)
orders[orders.order_id.isin(no_definidos.order_id)][["order_id", "order_status"]]

                               order_id  payment_sequential payment_type  \
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411  00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
94427  c8c528189310eaa44a745b8d9d26908b                   1  not_defined   

       payment_installments  payment_value  
51280                     1            0.0  
57411                     1            0.0  
94427                     1            0.0  


,order_id,order_status
1130,00b1cb0320190ca0daa2c88b35206009,canceled
39919,4637ca194b6387e2d538dc89b124b0ee,canceled
40235,c8c528189310eaa44a745b8d9d26908b,canceled


In [127]:
payments[payments.payment_installments == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [129]:
# .loc[condición, columna] = valor: donde se cumple la condición, cambia esa columna
payments.loc[payments.payment_installments == 0, "payment_installments"] = 1

print((payments.payment_installments == 0).sum())   # debe salir 0
print(len(payments))                                # debe seguir siendo 103886

payments.to_csv(CLEAN / "payments.csv", index=False)

0
103886


# Tabla products

In [149]:
products = pd.read_csv(RAW / "olist_products_dataset.csv")
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [151]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

## Renombramos columnas mal escritas:

In [154]:
# rename(columns={"nombre_viejo": "nombre_nuevo"})
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

products.columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')

## Categorías nulas

No las podemos borrar, rellenamos y le damos el nombre sem_categoria

In [157]:
print(products.product_category_name.isna().sum())    

products["product_category_name"] = products["product_category_name"].fillna("sem_categoria")

print(products.product_category_name.isna().sum())     

610
0


## Peso 0

Pasamos a nulo. Un producto no puede pesar 0y esto puede alterar medias y otros calculos

In [166]:
print((products.product_weight_g == 0).sum())       

products.loc[products.product_weight_g == 0, "product_weight_g"] = np.nan

print((products.product_weight_g == 0).sum())       

4
0


In [170]:
print(len(products))                    # debe seguir siendo 32951

print(products.product_id.is_unique)    # True

print(products.isna().sum())

products.to_csv(CLEAN / "products.csv", index=False)

32951
True
product_id                      0
product_category_name           0
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                6
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


# Tabla translation

In [175]:
translation = pd.read_csv(RAW / "product_category_name_translation.csv")
print(len(translation))    
translation.tail()

71


,product_category_name,product_category_name_english
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes
70,seguros_e_servicos,security_and_services


## Filas nuevas

Añadimos 3 filas a mano, ya que tenemos 71 filas pero nuestros products usan 73 + la nueva que añadimos de sem_categoria

In [179]:
# Una tabla pequeña con las 3 filas nuevas (mismas columnas que la original)
nuevas = pd.DataFrame({
    "product_category_name": [
        "pc_gamer",
        "portateis_cozinha_e_preparadores_de_alimentos",
        "sem_categoria",
    ],
    "product_category_name_english": [
        "pc_gamer",
        "portable_kitchen_and_food_preparers",
        "uncategorized",
    ],
})

# pd.concat apila tablas una debajo de otra; ignore_index=True renumera las filas de 0 a n-1
translation = pd.concat([translation, nuevas], ignore_index=True)
print(len(translation))      # 74

74


In [181]:
# Categorías de products (la tabla limpia que ya tienes en memoria) que NO están en translation
sin_traduccion = products[~products.product_category_name.isin(translation.product_category_name)]
print(len(sin_traduccion))                                # debe ser 0
print(translation.product_category_name.is_unique)        # True

0
True


In [183]:
translation.to_csv(CLEAN / "translation.csv", index=False)

# Tabla geolocation

In [187]:
geolocation = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
print(geolocation.shape)       # (1000163, 5)

(1000163, 5)


## Quitar duplicados exactos

In [191]:
geolocation = geolocation.drop_duplicates()
print(len(geolocation))       

738332


## Quitar coordenadas fuera de Brasil

In [194]:
# Rectángulo aproximado de Brasil: latitud entre -34 y 6, longitud entre -74 y -34
dentro_de_brasil = geolocation.geolocation_lat.between(-34, 6) & geolocation.geolocation_lng.between(-74, -34)

print((~dentro_de_brasil).sum())      # cuántas se descartan (~ invierte True/False)
geolocation = geolocation[dentro_de_brasil]
print(len(geolocation))

33
738299


## Una fila por código postal

In [197]:
geolocation = (
    geolocation
    .groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
    .median()
    .reset_index()
)
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,1001,-23.549951,-46.634027
1,1002,-23.548228,-46.635247
2,1003,-23.548977,-46.635313
3,1004,-23.549550,-46.634771
4,1005,-23.549763,-46.636100


In [201]:
print(len(geolocation))                                     
print(geolocation.geolocation_zip_code_prefix.is_unique)     
geolocation.to_csv(CLEAN / "geolocation.csv", index=False)

19010
True


# Tabla customers

In [204]:
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")

print(customers.isna().sum().sum())             # nulos totales: 0
print(customers.duplicated().sum())             # filas duplicadas: 0
print(customers.customer_id.is_unique)          # True
print(customers.customer_state.nunique())       # 27 estados

# ciudades que no estén ya en minúsculas y sin espacios sobrantes: 0
print((customers.customer_city != customers.customer_city.str.lower().str.strip()).sum())

0
0
True
27
0


In [206]:
customers.to_csv(CLEAN / "customers.csv", index=False)

# Tabla sellers

In [209]:
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")

# ciudades con algún carácter distinto de letras minúsculas y espacio
raras = sellers.seller_city.str.contains(r"[^a-z ]", regex=True)
print(raras.sum())
sellers.seller_city[raras].head(15)

34


78                                    lages - sc
237                                 auriflama/sp
246                        sao paulo / sao paulo
360                                   são paulo
476                        santa barbara d´oeste
517                                     04482255
551     novo hamburgo, rio grande do sul, brasil
622                               cariacica / es
707                           sao miguel d'oeste
826                               sao paulo - sp
869                                       sbc/sp
874               arraial d'ajuda (porto seguro)
945                        santo andre/sao paulo
1004                                     sp / sp
1159                              maua/sao paulo
Name: seller_city, dtype: object

Tenemos que cambiar los caracteres raros

## Quitar tildes y normalizar apóstrofo

In [213]:
sellers["seller_city"] = (
    sellers["seller_city"]
    .str.replace("´", "'", regex=False)   # apóstrofo raro -> normal
    .str.normalize("NFKD")                # separa cada letra de su tilde
    .str.encode("ascii", errors="ignore") # al pasar a ASCII, las tildes desaparecen
    .str.decode("ascii")                  # y volvemos a texto
)

## Nos quedamos con lo que hay antes del separador

In [216]:
# Separadores: "/", "\", ",", "(" o " - " (guion con espacios)
sellers["seller_city"] = (
    sellers["seller_city"]
    .str.split(r"\s*[/\\,(]\s*|\s+-\s+", n=1, regex=True)
    .str[0]
    .str.strip()
)

## Casos sueltos a mano

In [219]:
arreglos = {
    "andira-pr": "andira",
    "sbc": "sao bernardo do campo",
    "sp": "sao paulo",
    "04482255": "rio de janeiro",                # deducido del prefijo de código postal y del estado RJ
    "vendas@creditparts.com.br": "maringa",      # deducido del prefijo de código postal y del estado PR
}
sellers["seller_city"] = sellers["seller_city"].replace(arreglos)

In [221]:
raras = sellers.seller_city.str.contains(r"[^a-z '\-]", regex=True)
print(raras.sum())                        # debe salir 0
print(sellers.seller_city.nunique())      # debe salir 586
print(len(sellers))                       # debe seguir siendo 3095

# los arreglos a mano: ¿tiene sentido el estado de esos vendedores?
sellers[sellers.seller_city.isin(["maringa", "rio de janeiro"])].seller_state.value_counts()

sellers.to_csv(CLEAN / "sellers.csv", index=False)

0
586
3095
